# Chapter 29 — Multivariate Statistics
*Tier 3: Students & Data Scientists*

## 🎯 Learning Objectives
- Work with multivariate normal distributions
- Apply PCA for dimensionality reduction
- Understand MANOVA, discriminant analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_digits
from sklearn.preprocessing import StandardScaler
import seaborn as sns

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Multivariate Normal Distribution

$$\mathbf X \sim N_p(\boldsymbol\mu, \boldsymbol\Sigma)$$

$$f(\mathbf x) = \frac{1}{(2\pi)^{p/2}|\boldsymbol\Sigma|^{1/2}}
\exp\left(-\frac{1}{2}(\mathbf x - \boldsymbol\mu)^T \boldsymbol\Sigma^{-1} (\mathbf x - \boldsymbol\mu)\right)$$

Key properties:
- Marginals are univariate normals
- Conditional distributions are also normal
- $\mathbf{X}^T \boldsymbol\Sigma^{-1}\mathbf X \sim \chi^2_p$ (Mahalanobis distance)

In [ ]:
# Bivariate normal: effect of correlation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, rho in zip(axes, [-0.8, 0.0, 0.8]):
    Sigma = np.array([[1, rho], [rho, 1]])
    mu_bv = np.array([0, 0])
    data_bv = rng.multivariate_normal(mu_bv, Sigma, 500)
    ax.scatter(data_bv[:,0], data_bv[:,1], alpha=0.3, s=15)
    # Plot contour of PDF
    x_g = np.linspace(-3.5, 3.5, 80)
    X1, X2 = np.meshgrid(x_g, x_g)
    pos = np.dstack((X1, X2))
    rv = stats.multivariate_normal(mu_bv, Sigma)
    ax.contour(X1, X2, rv.pdf(pos), levels=6, colors="red", linewidths=1)
    ax.set_title(f"ρ={rho}")
    ax.set_xlabel("X₁"); ax.set_ylabel("X₂")
plt.suptitle("Bivariate Normal — Effect of Correlation", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Principal Component Analysis

PCA finds orthogonal directions of maximum variance.

1. Compute covariance matrix $\mathbf S$
2. Eigen-decompose: $\mathbf S = \mathbf V \mathbf\Lambda \mathbf V^T$
3. Project: $\mathbf Z = \mathbf X \mathbf V_k$ (first $k$ eigenvectors)

Proportion of variance explained by PC $j$:
$$\frac{\lambda_j}{\sum_i \lambda_i}$$

In [ ]:
# PCA on Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_scaled = StandardScaler().fit_transform(X_iris)

pca = PCA()
pca.fit(X_scaled)
explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, 5), explained*100, color="steelblue")
axes[0].plot(range(1, 5), np.cumsum(explained)*100, "ro-")
axes[0].set_xlabel("Principal Component"); axes[0].set_ylabel("Variance Explained (%)")
axes[0].set_title("Scree Plot — Iris PCA")

X_pca = pca.transform(X_scaled)
colors = ["red", "blue", "green"]
for cls in range(3):
    mask = y_iris == cls
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], alpha=0.7, s=30,
                    label=iris.target_names[cls], color=colors[cls])
axes[1].set_xlabel(f"PC1 ({explained[0]*100:.1f}%)"); axes[1].set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
axes[1].set_title("PCA — Iris (2 Components)"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"PC1+PC2 explain {(explained[0]+explained[1])*100:.1f}% of variance")

In [ ]:
# Mahalanobis distance for outlier detection
Sigma_est = np.cov(X_iris.T)
mu_est = X_iris.mean(axis=0)
Sigma_inv = np.linalg.inv(Sigma_est)
mahal_dist = np.array([
    np.sqrt((x - mu_est) @ Sigma_inv @ (x - mu_est))
    for x in X_iris
])
threshold = np.sqrt(stats.chi2.ppf(0.975, df=4))
outliers = np.where(mahal_dist > threshold)[0]
print(f"Outliers (Mahalanobis > {threshold:.2f}): {len(outliers)} samples")
print(f"Indices: {outliers}")

plt.figure(figsize=(10, 4))
plt.plot(mahal_dist, "b.", markersize=4)
plt.axhline(threshold, color="red", linestyle="--", label=f"χ² threshold (97.5%)")
plt.scatter(outliers, mahal_dist[outliers], color="red", s=60, zorder=5, label="Outliers")
plt.xlabel("Sample index"); plt.ylabel("Mahalanobis distance")
plt.title("Outlier Detection via Mahalanobis Distance")
plt.legend(); plt.tight_layout(); plt.show()